In [1]:
import json
import glob
import polars as pl
import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow
from datetime import datetime


In [2]:
def load_ndjson(file_path):
    with open(file_path, 'r') as file:
        data = [json.loads(line) for line in file]
    return data

def convert_time_to_seconds(time_str):
    # Extract the time part (hhmmss.SSS)
    time_part = time_str.split('-')[1]
    # Parse the time part to a datetime object
    time_obj = datetime.strptime(time_part, '%H%M%S.%f')
    # Convert to seconds
    seconds = time_obj.hour * 3600 + time_obj.minute * 60 + time_obj.second + time_obj.microsecond / 1_000_000
    return seconds

def convert_since_creation_to_seconds(since_creation_str):
    return int(since_creation_str.replace('ms', '')) / 1000


## Import data
The raw DuckSoup logs (`data/real_data`, about 800 MB) are not shared. The notebook reads `data/network_logs.csv.gz` (2 MB), which keeps the logs plotted below.
To rebuild `network_logs.csv.gz` from the raw logs, set `parse_logs = True`.

In [3]:
#True : parse the raw logs in ../data/real_data and write ../data/network_logs.csv.gz; False : only read network_logs.csv.gz
parse_logs = False


In [4]:
#Parse the raw logs (only if parse_logs is True)
dfs = []

if parse_logs:
    for file in glob.glob("../data/real_data/*/*/*.log"):
        if os.stat(file).st_size == 0:
            print("File is empty, skipping:", file)
            continue
        file_parts  = file.split("/")[-1].split("-")
        round_nb = file.split("/")[-1].split("-")[0]
        dyad = file.split("/")[-1].split("-")[1]
        sid = file.split("/")[-3]

        data = load_ndjson(file)
        df = pd.DataFrame(data)
        df['time_seconds'] = df['time'].apply(convert_time_to_seconds)

        dfs.append(df)

In [5]:
if parse_logs:
    all_data_df = pd.concat(dfs)
    all_data_df['sinceCreation_seconds']     = all_data_df['sinceCreation'].str.replace('ms', '').astype(float) / 1000
    all_data_df['sinceCreation_downsampled'] = (all_data_df['sinceCreation_seconds'] // 10) * 10

    #Keep the track logs plotted below (bit rates, fps, resolution...) and the columns used
    plotted_logs = ["video_in_bitrate", "video_out_bitrate", "audio_in_bitrate", "client_video_fps_updated", "audio_out_bitrate", "client_keyframe_encoded_count_updated", "client_keyframe_decoded_count_updated", "client_pli_received_count_updated", "client_fir_received_count_updated", "client_video_resolution_updated"]
    all_data_df = all_data_df.loc[(all_data_df["context"] == "track") & all_data_df["message"].isin(plotted_logs)]

    #Keep the sessions analysed (the sessions of all_data_df.csv, written by behaviour/Behavior.ipynb)
    sessions = pd.read_csv("../data/behavior/all_data_df.csv")["sid"].str.replace(":", "").unique()
    all_data_df = all_data_df.loc[all_data_df["namespace"].isin(sessions)]
    all_data_df = all_data_df[["namespace", "interaction", "user", "context", "message", "value", "sinceCreation_seconds", "sinceCreation_downsampled"]]
    all_data_df.to_csv("../data/network_logs.csv.gz", index=False)


In [6]:
#Import the network logs
all_data_df = pd.read_csv("../data/network_logs.csv.gz", dtype={"value": str})
all_data_df.head()


,namespace,interaction,user,context,message,value,sinceCreation_seconds,sinceCreation_downsampled
0,mkprolific_participant_real_22,4-p2p8,p8,track,video_in_bitrate,194,5.962,0.0
1,mkprolific_participant_real_22,4-p2p8,p8,track,audio_in_bitrate,45,5.962,0.0
2,mkprolific_participant_real_22,4-p2p8,p8,track,video_out_bitrate,0,5.963,0.0
3,mkprolific_participant_real_22,4-p2p8,p8,track,audio_out_bitrate,0,5.963,0.0
4,mkprolific_participant_real_22,4-p2p8,p2,track,video_in_bitrate,199,5.963,0.0


In [7]:
# Import behavioral data and prepare for merge
behavior_df = pl.read_csv("../data/behavior/all_data_df.csv")
behavior_df.head()

remove_colons = lambda s: s.replace(":", "")

# Apply the lambda function to the specified column
behavior_df = behavior_df.with_columns(
    behavior_df['sid'].map_elements(remove_colons, return_dtype=pl.Utf8).alias('sid'),
    behavior_df['user_id'].map_elements(remove_colons, return_dtype=pl.Utf8).alias('user_id'),
    behavior_df['other_id'].map_elements(remove_colons, return_dtype=pl.Utf8).alias('other_id')
)

behavior_df = behavior_df.pivot(index=["sid", "dyad", "user_id", "other_id", "participant_condition", "other_condition"], on="question_content", values="measure",aggregate_function="sum")

/var/folders/g5/mqbcmmn575z0s3c021r942c80000gp/T/ipykernel_4454/1856255819.py:9: PolarsInefficientMapWarning: 
Series.map_elements is significantly slower than the native series API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - s.map_elements(lambda s: ...)
with this one instead:
  + s.str.replace_all(':','',literal=True)

  behavior_df['sid'].map_elements(remove_colons, return_dtype=pl.Utf8).alias('sid'),
/var/folders/g5/mqbcmmn575z0s3c021r942c80000gp/T/ipykernel_4454/1856255819.py:10: PolarsInefficientMapWarning: 
Series.map_elements is significantly slower than the native series API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - s.map_elements(lambda s: ...)
with this one instead:
  + s.str.replace_all(':','',literal=True)

  behavior_df['user_id'].map_elements(remove_colons, return_dtype=pl.Utf8).alias('user_id'),
/var/folders/g5/mqbcmmn575z0s3c021r942c80000gp/T/ipykernel_4454/185

In [8]:
logs = ["video_in_bitrate", "video_out_bitrate", "audio_in_bitrate", "client_video_fps_updated", "audio_out_bitrate", "client_keyframe_encoded_count_updated", "client_keyframe_decoded_count_updated", "client_pli_received_count_updated", "client_fir_received_count_updated", "client_video_resolution_updated"]

for log in logs:
    df_analysis = all_data_df.copy()
    df_analysis = df_analysis.loc[df_analysis["context"] == "track"]
    df_analysis = df_analysis.loc[df_analysis["message"] == log]
    if log == "client_video_resolution_updated":
        df_analysis['width'] = [float(x.split("x")[0]) for x in df_analysis["value"].values]
        df_analysis['height'] = [float(x.split("x")[1]) for x in df_analysis["value"].values]
        df_analysis = df_analysis.groupby(["namespace", "interaction", "user", "sinceCreation_seconds"]).mean(numeric_only=True).reset_index()
        df_analysis = df_analysis.loc[df_analysis["sinceCreation_seconds"] < 250]

        #save weight
        g = sns.relplot(x="sinceCreation_downsampled", y= "width", data=df_analysis, kind="line", aspect=0.7, height=2)
        plt.savefig("plots/" + log + "_width_legend.pdf", bbox_inches='tight')
        g.set(xticklabels=[], yticklabels=[], title='',xlabel=None, ylabel=None)
        plt.savefig("plots/" + log + "_width.pdf", bbox_inches='tight')
        
        #save height
        g = sns.relplot(x="sinceCreation_downsampled", y= "height", data=df_analysis, kind="line", aspect=0.7, height=2)
        plt.savefig("plots/" + log + "_height_legend.pdf", bbox_inches='tight')
        g.set(xticklabels=[], yticklabels=[], title='',xlabel=None, ylabel=None)
        plt.savefig("plots/" + log + "_height.pdf", bbox_inches='tight')
    else:    
        df_analysis['value'] = [float(x) for x in df_analysis["value"].values]
        df_analysis = df_analysis.groupby(["namespace", "interaction", "user", "sinceCreation_seconds"]).mean(numeric_only=True).reset_index()
        df_analysis = df_analysis.loc[df_analysis["sinceCreation_seconds"] < 250]
        g = sns.relplot(x="sinceCreation_downsampled", y= "value", data=df_analysis, kind="line", aspect=0.7, height=2.5)
    
    plt.savefig("plots/" + log + "_legend.pdf", bbox_inches='tight')
    g.set(xticklabels=[], yticklabels=[], title='',xlabel=None, ylabel=None)
    plt.savefig("plots/" + log + ".pdf", bbox_inches='tight')

In [9]:
all_data_df.columns

Index(['namespace', 'interaction', 'user', 'context', 'message', 'value',
       'sinceCreation_seconds', 'sinceCreation_downsampled'],
      dtype='object')

In [10]:
df_analysis = all_data_df.copy()
df_analysis = df_analysis.loc[df_analysis["context"] == "track"]
df_analysis = df_analysis.loc[df_analysis["message"] == "video_in_bitrate"]
df_analysis.columns

Index(['namespace', 'interaction', 'user', 'context', 'message', 'value',
       'sinceCreation_seconds', 'sinceCreation_downsampled'],
      dtype='object')

In [11]:
df_analysis["value"]

0          194
4          199
19         153
22         171
33         164
          ... 
326434    1516
326442    1325
326446    1500
326456    1320
326459    1278
Name: value, Length: 54277, dtype: object